# ☁️ Prologis Financial Assistant — Cloud Deployment

This notebook documents the multi-cloud deployment pipeline:
1. AWS S3 — Model artifact storage
2. AWS SageMaker — Model endpoint deployment
3. GCP Vertex AI — Gemini chatbot integration
4. Endpoint testing and validation

## 1. Setup

In [1]:
import boto3, json, os, tarfile, tempfile, shutil
import pandas as pd
import numpy as np

AWS_REGION   = 'us-east-2'
AWS_ACCOUNT  = '362612348837'
S3_BUCKET    = 'prologis-financial-assistant-362612348837'
IAM_ROLE_ARN = 'arn:aws:iam::362612348837:role/SageMakerRole'
GCP_PROJECT  = 'project-f47a88c6-9a3a-4b51-903'

print('AWS Region  :', AWS_REGION)
print('S3 Bucket   :', S3_BUCKET)
print('GCP Project :', GCP_PROJECT)

AWS Region  : us-east-2
S3 Bucket   : prologis-financial-assistant-362612348837
GCP Project : project-f47a88c6-9a3a-4b51-903


## 2. AWS S3 — Model Upload

In [2]:
# List S3 bucket contents
s3 = boto3.client('s3', region_name=AWS_REGION)

print(f'Objects in s3://{S3_BUCKET}/')
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=S3_BUCKET):
    for obj in page.get('Contents', []):
        size_mb = obj['Size'] / (1024*1024)
        print(f"  {obj['Key']:<60} {size_mb:.2f} MB")

Objects in s3://prologis-financial-assistant-362612348837/
  inference/regression/inference.py                            0.00 MB
  models/classification/classification_model.tar.gz            0.00 MB
  models/classification/classification_model_fixed.tar.gz      0.00 MB
  models/classification/classification_model_v3.tar.gz         0.00 MB
  models/regression/regression_model.tar.gz                    29.39 MB
  models/regression/regression_model_fixed.tar.gz              29.39 MB
  models/regression/regression_model_v3.tar.gz                 0.00 MB


In [3]:
# Verify model tarballs exist
for key in ['models/regression/regression_model_fixed.tar.gz',
            'models/classification/classification_model_fixed.tar.gz']:
    try:
        r = s3.head_object(Bucket=S3_BUCKET, Key=key)
        print(f'✅ {key} ({r["ContentLength"]/1024/1024:.1f} MB)')
    except:
        print(f'❌ {key} NOT FOUND')

✅ models/regression/regression_model_fixed.tar.gz (29.4 MB)
✅ models/classification/classification_model_fixed.tar.gz (0.0 MB)


## 3. AWS SageMaker — Endpoint Status

In [9]:
# Check endpoint status
sm = boto3.client('sagemaker', region_name=AWS_REGION)

endpoints = ['prologis-regression-endpoint', 'prologis-classification-endpoint']
for ep in endpoints:
    try:
        r = sm.describe_endpoint(EndpointName=ep)
        status = r['EndpointStatus']
        symbol = '✅' if status == 'InService' else '⚠️'
        print(f'{symbol} {ep}: {status}')
        if status == 'Failed':
            print(f'   Reason: {r.get("FailureReason", "Unknown")}')
    except sm.exceptions.ClientError:
        print(f'❌ {ep}: Not found')

✅ prologis-regression-endpoint: InService
⚠️ prologis-classification-endpoint: Creating


In [5]:
# SageMaker IAM Role verification
iam = boto3.client('iam', region_name=AWS_REGION)
try:
    r = iam.get_role(RoleName='SageMakerRole')
    print('✅ SageMaker IAM Role exists')
    role = r['Role']
    print('   ARN    :', role.get('Arn', role.get('RoleArn', 'N/A')))
    print('   Created:', role.get('CreateDate', 'N/A'))
except Exception as e:
    print(f'❌ IAM Role error: {e}')

✅ SageMaker IAM Role exists
   ARN    : arn:aws:iam::362612348837:role/SageMakerRole
   Created: 2026-06-09 06:26:22+00:00


## 4. Local Model Inference (Fallback)

In [8]:
# Test local model inference
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
from model_loader import predict_revenue, predict_risk

# Test regression
reg_features = [5.0, 20.0, 6.0, 1.1, 1500.0, 3.0, 37.8, -122.4]
reg_result   = predict_revenue(reg_features)
print('Regression prediction:', reg_result)

# Test classification
clf_features = [35, 4, 1, 2, 0, 5000, 0, 0, 0, 3, 5, 500, 1, -1, 2, 2]
clf_result   = predict_risk(clf_features)
print('Classification prediction:', clf_result)

C:\Users\ktvab\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Regression prediction: {'prediction': 269560.99999999994}
Classification prediction: {'prediction': ['Low'], 'probabilities': [[0.0, 1.0]]}


In [9]:
# Test multiple scenarios
scenarios = {
    'High Value Property (SF)':  [8.0, 5.0, 8.0, 2.0, 200.0, 1.5, 37.8, -122.4],
    'Mid Value Property (LA)':   [5.0, 20.0, 6.0, 1.5, 800.0, 3.0, 34.0, -118.2],
    'Low Value Property (Inland)':[2.0, 40.0, 4.0, 1.2, 2000.0, 5.0, 35.0, -119.0],
}

print('Revenue Forecast Scenarios:')
print('-' * 50)
for name, features in scenarios.items():
    result = predict_revenue(features)
    rev    = result.get('prediction', 0)
    print(f'{name:<35} ${rev:>12,.0f}')

Revenue Forecast Scenarios:
--------------------------------------------------


C:\Users\ktvab\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


High Value Property (SF)            $     414,114


C:\Users\ktvab\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Mid Value Property (LA)             $     243,943
Low Value Property (Inland)         $      76,762


C:\Users\ktvab\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 5. GCP Vertex AI — Chatbot Integration

In [10]:
# Test Vertex AI connection
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project=GCP_PROJECT, location='us-central1')
model = GenerativeModel('gemini-2.5-flash')

# Test with a Prologis financial query
prompt = '''You are a financial analyst for Prologis Inc., a leading industrial REIT.
Answer briefly: What are the key drivers of NOI growth for industrial REITs?'''

response = model.generate_content(prompt)
print('Vertex AI Response:')
print('-' * 50)
print(response.text)

C:\Users\ktvab\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\ktvab\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
C:\Users\ktvab\AppData\Local\Programs\Python

Vertex AI Response:
--------------------------------------------------
For an industrial REIT like Prologis, the key drivers of Net Operating Income (NOI) growth are:

1.  **Market Rent Growth:** Strong demand (e-commerce, supply chain resilience, inventory re-stocking) coupled with limited new supply drives significant increases in market rents.
2.  **High Occupancy Rates:** Maximizing leased space minimizes vacancy costs and maximizes rental income.
3.  **Contractual Lease Escalations:** Built-in annual rent increases (e.g., 2-3% or CPI-linked) within existing long-term leases.
4.  **Effective Operating Cost Management:** Efficiently managing property expenses like utilities, maintenance, property taxes, and insurance.
5.  **Value-Add Developments & Acquisitions:** Developing new, state-of-the-art logistics facilities or acquiring well-located, under-market properties to lease at higher rates.


In [11]:
# Test with SEC data context
import json

with open('../data/sec_mock.json') as f:
    sec_data = json.load(f)

prompt = f'''You are a financial analyst for Prologis Inc.
Based on this SEC filing data: {json.dumps(sec_data)[:500]}
Provide a brief summary of the key financial metrics.'''

response = model.generate_content(prompt)
print('SEC Data Analysis:')
print('-' * 50)
print(response.text)

SEC Data Analysis:
--------------------------------------------------
As a financial analyst for Prologis Inc. (PLD), based on the latest SEC filing data, primarily the Q4 2024 10-K filed on February 13, 2025, here is a brief summary of key financial metrics:

*   **Revenue:** Prologis reported strong revenue of **$2.65 billion** for the period (likely representing the full fiscal year given the 10-K filing type).
*   **Net Income:** The company achieved substantial net income of **$1.12 billion**, indicating robust profitability.
*   **Earnings Per Share (EPS):** Shareholders saw a healthy **$3.22** in earnings per share.
*   **Operating Expenses:** Operating expenses for the period were **$1.29 billion**.
*   **Balance Sheet Strength:** Prologis maintains a strong financial position with **total assets of $92.0 billion** against **total liabilities of $38.0 billion**, reflecting significant financial stability and equity.


## 6. Database Connection Verification

In [12]:
import psycopg2
conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='realestate_db',
    user='postgres', password='Ktvab@2k'
)

df = pd.read_sql('SELECT * FROM properties LIMIT 5', conn)
print(f'✅ Database connected — {len(pd.read_sql("SELECT * FROM properties", conn))} properties')
df

✅ Database connected — 22 properties


C:\Users\ktvab\AppData\Local\Temp\ipykernel_25664\1686781895.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('SELECT * FROM properties LIMIT 5', conn)
C:\Users\ktvab\AppData\Local\Temp\ipykernel_25664\1686781895.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(f'✅ Database connected — {len(pd.read_sql("SELECT * FROM properties", conn))} properties')


,property_id,address,metro_area,sq_footage,property_type
0,1,233 S Wacker Dr,Chicago,1850000,Office
1,2,350 N Orleans St,Chicago,420000,Industrial
2,3,1 Infinite Loop,San Francisco,980000,Office
3,4,2000 W Fulton St,Chicago,310000,Industrial
4,5,555 Market St,San Francisco,670000,Retail


## 7. Architecture Summary

| Component | Service | Status |
|---|---|---|
| Model Storage | AWS S3 | ✅ Active |
| ML Inference | Local (model_loader.py) | ✅ Active |
| SageMaker Endpoints | AWS SageMaker | ⚠️ Created (sklearn version mismatch) |
| AI Chatbot | GCP Vertex AI (Gemini 2.5 Flash) | ✅ Active |
| Database | PostgreSQL (local) | ✅ Active |
| Frontend | Streamlit | ✅ Active |

### Known Issue — SageMaker Endpoints
Models were trained with scikit-learn 1.2.2 but the SageMaker SKLearn container 1.2-1 has a compatibility issue with the 144MB Random Forest model (ping health check timeout). The models run locally via `model_loader.py` as a reliable fallback.

### Multi-Cloud Architecture
- **AWS**: S3 (storage), SageMaker (deployment infrastructure), IAM (security)
- **GCP**: Vertex AI (Gemini 2.5 Flash chatbot, generative AI)